In [1]:
import numpy as np 

In [112]:
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            self._column_num = len(pauli_word[0])
            self._row_num = len(pauli_word)
        
            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)
    
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._column_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num
    
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z
                    
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    def copy(self) -> "Cirq_Tableau":
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def __copy__(self) -> "Cirq_Tableau":
        return self.copy()
        

    def __str__(self):
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)

In [113]:
def actions(num_qubits: int):
    S = [("S", i) for i in range(num_qubits)]
    H = [("H", i) for i in range(num_qubits)]
    CX = []
    for i in range(num_qubits):
        if i != (num_qubits-1):
            for j in range((i+1),num_qubits):
                CX.append(("CX", i, j))
                CX.append(("CX", j, i))
    return S + H + CX

In [114]:
word = ["XX", "-ZZ"]

In [115]:
tableau = Cirq_Tableau(word)

In [116]:
print(tableau)
tab = tableau.copy()
print(tab)

[[1 1 0 0 0]
 [0 0 1 1 1]]
[[1 1 0 0 0]
 [0 0 1 1 1]]


In [49]:
tableau.apply_H(0)

In [50]:
print(tableau)

[[0 1 1 0 0]
 [1 0 0 1 1]]


In [51]:
tableau.apply_H(0)
print(tableau)

[[1 1 0 0 0]
 [0 0 1 1 1]]


In [52]:
tableau.apply_CX(0,1)

In [53]:
print(tableau)

[[1 0 0 0 0]
 [0 0 0 1 1]]


In [54]:
tableau.apply_CX(0,1)
print(tableau)

[[1 1 0 0 0]
 [0 0 1 1 1]]


In [55]:
tableau.apply_S(0)
print(tableau)

[[1 1 1 0 0]
 [0 0 1 1 1]]


In [56]:
tableau.apply_S(0)
print(tableau)


[[1 1 0 0 1]
 [0 0 1 1 1]]
